# Lab 00 · Setup and model inventory
**~5 minutes · costs about $0.001**

Confirms your environment works and shows you the current model IDs, so the
other labs use real strings instead of ones copied from a page that may have
drifted.

> **Before you start.** These labs call `api.anthropic.com`. If you are on a
> locked-down corporate network with egress restrictions, this will fail with a
> connection error, not an auth error. Run them from a personal machine if so.

Install once:

```
pip install anthropic jupyter
export ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import os, sys
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("warning: ANTHROPIC_API_KEY is not set; the client will fail unless another credential source is configured")
import anthropic
client = anthropic.Anthropic()
print("python", sys.version.split()[0])
print("anthropic sdk", anthropic.__version__)

## The Models API

Query limits rather than hardcoding them. `max_input_tokens` (the context
window) and `max_tokens` (the ceiling for the `max_tokens` request parameter,
i.e. output) are two different numbers, and mixing them up is a classic
model-selection mistake. The list comes back newest first.

In [ ]:
for m in client.models.list():
    print(f"{m.id:<22} {m.display_name:<20} in={getattr(m,'max_input_tokens','?')!s:<9} out={getattr(m,'max_tokens','?')}")

**Set your working models.** The study site standardises on
`claude-sonnet-5-5` as the everyday model and `claude-haiku-4-5` for cheap,
high-volume calls (`src/data/facts.json`). Edit only if the list above
disagrees.

`text_of` is the one helper every lab defines: it joins the text blocks of a
response. Never read `content[0].text`; on current models the first block is
often a `thinking` block, which has no `.text`.

In [ ]:
MODEL = "claude-sonnet-5-5"     # edit only if the list above disagrees
CHEAP = "claude-haiku-4-5"

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

r = client.messages.create(model=MODEL, max_tokens=1024,
        messages=[{"role":"user","content":"Reply with exactly: ok"}])
print(text_of(r), "|", r.stop_reason, "|", r.usage.input_tokens, "in /", r.usage.output_tokens, "out")
print("block types:", [b.type for b in r.content])

### What you should have seen
A model list, and `ok | end_turn | N in / M out`.

Why `max_tokens=1024` for a two-letter answer: Sonnet 5.5 runs adaptive
thinking by default, and thinking tokens count against `max_tokens` and are
billed as output. With a tiny value such as 10 the response can stop with
`stop_reason == "max_tokens"` before any text block exists. `max_tokens` is a
ceiling, not a target, so a generous value costs nothing extra.

If you got `APIConnectionError`, it is the network, not your key.
If you got `AuthenticationError`, the key is wrong or unset.